# Preprocessing

Complete `moving_average`, `feature`, `normalized_distances`, and `predict`, then run their checks and plots. Replace each `NotImplementedError`; keep the supplied validation.

Run cells in order. Bundled synthetic recordings cover four gestures with rest, movement, hold, and release. Results describe the simulation only.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'calibration_pipeline/eflesh_calibration/knn.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the GuaqBand checkout.')
LAB = ROOT/'notebooks/signals_basics/preprocessing'
sys.path.insert(0, str(LAB))
sys.path.insert(0, str(ROOT))
import lab_support as lab
import importlib
importlib.reload(lab)
from calibration_pipeline.eflesh_calibration.knn import (
    FILTER_SAMPLES, RAW_WINDOW_SAMPLES, WINDOW_SAMPLES, SAMPLE_HZ, FEATURE_COUNT,
)
from calibration_pipeline.eflesh_calibration.sensor import CHANNEL_COUNT

USE_SYNTHETIC = True
trials = (lab.load_synthetic_trials(LAB/'synthetic/gesture_trials.npz') if USE_SYNTHETIC
          else lab.load_trials(LAB/'recordings/manifest.csv'))
print(f"{len(trials)} {'SYNTHETIC' if USE_SYNTHETIC else 'recorded'} trials · 50 Hz · 12 channels")
print('Per gesture: 6 train, 2 tune, 2 test.' if USE_SYNTHETIC else 'Splits from manifest.')


## 1 · Moving average

Input: `(N, 12)`, N ≥ 5. Output: `(N−4, 12)`.

Average five consecutive samples per channel. Use complete windows only, with no padding or future values. Each output belongs to the last input timestamp it uses.

Predict the outputs for `[0, 0, 0, 0, 5, 5]` before coding.

In [ ]:
def moving_average(samples: np.ndarray) -> np.ndarray:
    """Return causal five-sample averages without padding or future samples."""
    samples = np.asarray(samples, dtype=float)
    if (
        samples.ndim != 2
        or samples.shape[0] < FILTER_SAMPLES
        or samples.shape[1] != CHANNEL_COUNT
    ):
        raise ValueError(
            f"samples must contain at least {FILTER_SAMPLES} rows of 12 sensor values"
        )
    if not np.isfinite(samples).all():
        raise ValueError("moving-average input must be finite")

    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement moving_average")


**Hints:** rows are time; columns are channels. Check your first and last complete windows.

In [ ]:
AVERAGE_READY = lab.verify_average(moving_average)

### Smoothing

Edit the settings below, then press **Shift+Enter**. Try `WINDOW_START_S = 2.2` for movement or `3.0` for a held pose. `None` also selects the held pose. The outlined heatmap column is the selected channel.

In [ ]:
TRAIN_TRIAL_INDEX = 0
CHANNEL = 0
WINDOW_START_S = None  # None selects the start of the held pose
lab.plot_average_steps(trials, moving_average, TRAIN_TRIAL_INDEX, CHANNEL, WINDOW_START_S)


## 2 · Features

Inputs: raw window `(19, 12)` and relaxed baseline `(12,)`. Output: `(24,)`.

Call your moving average. Return 12 baseline-relative signed means followed by 12 baseline-relative RMS values, preserving channel order.

Predict mean and RMS for constant changes of +3 and −3 µT.

In [ ]:
def feature(raw_window: np.ndarray, baseline: np.ndarray) -> np.ndarray:
    """Smooth 19 raw frames, then return 15-frame signed mean and RMS features."""
    raw_window = np.asarray(raw_window, dtype=float)
    baseline = np.asarray(baseline, dtype=float)
    if raw_window.shape != (RAW_WINDOW_SAMPLES, CHANNEL_COUNT):
        raise ValueError("window must contain rows of 12 sensor values")
    if baseline.shape != (CHANNEL_COUNT,):
        raise ValueError("baseline must contain 12 sensor values")
    if not np.isfinite(raw_window).all() or not np.isfinite(baseline).all():
        raise ValueError("feature input must be finite")

    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement feature")

**Hint:** Use the relaxed baseline.

In [ ]:
FEATURES_READY = lab.verify_feature(feature, moving_average, AVERAGE_READY)

### Feature stages

Compare the smoothed signal, relaxed reference, baseline-relative signal, and your 24 features.

In [ ]:
lab.plot_feature_steps(trials, moving_average, feature, TRAIN_TRIAL_INDEX, CHANNEL)

## 3 · KNN

Input: model with `(N, 24)` training features and a `(24,)` query. Output: `(N,)` distances.

Standardize using the saved `feature_mean` and `feature_scale`. Return Euclidean distance to each training example. Do not fit new scales to the query.

In [ ]:
def normalized_distances(model: dict, sample: np.ndarray) -> np.ndarray:
    """Return one standardized Euclidean distance per training example."""
    training = np.array(model['features'])
    mean = np.array(model['feature_mean'])
    scale = np.array(model['feature_scale'])

    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement normalized_distances")

**Hint:** each distance combines differences across all 24 features. Keep training-row order.

In [ ]:
DISTANCES_READY = lab.verify_distances(normalized_distances)

### Vote

Implement `predict(model, sample)` → one gesture label. Call your distance function and use `model['k']` nearest examples (normally 3).

The most frequent label wins. For tied votes, choose the nearest tied label. Equal distances retain training-row order.

In [ ]:
def predict(model: dict, sample: np.ndarray) -> str:
    """Predict a gesture from the nearest standardized training examples."""
    model_labels = model['labels']
    
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement predict")

**Hint:** think in three steps: distances, neighbor order, label counts. Predict the vote for `[fist, rest, fist]`.

In [ ]:
KNN_READY = lab.verify_predict(predict, normalized_distances, DISTANCES_READY)

### See your neighbors

Change `QUERY_ROW` or `FEATURE_AXES` and rerun. The scatter shows two features; neighbors use all 24. The query is an illustration, not an accuracy test.

In [ ]:
QUERY_ROW = 0
FEATURE_AXES = (0, 1)
knn_plot_model = None
if FEATURES_READY:
    knn_plot_model = lab.fit_training(lab.build_examples(trials, feature))
lab.plot_knn_steps(normalized_distances, predict, model=knn_plot_model,
                   query_row=QUERY_ROW, axes=FEATURE_AXES)